# Results for model: microsoft/phi-3-medium-4k-instruct

In [1]:
import polars as pl

import pandas as pd

import xgboost as xgb

from xgboost import DMatrix

import numpy as np


# Constants

TOP_QUANTILE = 0.15

ROLLING_WINDOW = 'date_id'


# Load the dataset

df = pl.read_parquet('./jane_street_data/train.parquet')

df = df.with_column(pl.col('date_id').cummax().alias('cumulative_date_id'))


# Feature engineering: calculating the global top quantile feature

top_quantile_date_id = df.groupby('cumulative_date_id')['feature_00'].apply(

    lambda x: x.rank(pct=True).lt(1 - TOP_QUANTILE)

).max().alias('top_quantile_feature_00')

df = df.join(top_quantile_date_id)


# Prepare the dataset for XGBoost

target = df['responder_6']

features = df.drop(['cumulative_date_id', 'feature_00', 'responder_6']).rename(columns={'top_quantile_feature_00': 'feature_01'})


# Train-validation split

train_df = df[df['date_id'] < df['date_id'].max()]

val_df = df[df['date_id'] == df['date_id'].max()]


# Construct XGBoost matrices

dtrain = DMatrix(train_df[features.columns.tolist()], label=train_df[target.name])

dval = DMatrix(val_df[features.columns.tolist()], label=val_df[target.name])


# Train with XGBoost

params = {

    'objective': 'reg:squarederror',

    'colsample_bytree': 0.3,

    'learning_rate': 0.1,

    'max_depth': 5,

    'alpha': 10

}


num_round = 20

bst = xgb.train(params, dtrain, num_round)


# Predict on validation set

preds = bst.predict(dval)


# Calculate error metric (RMSE)

rmse = np.sqrt(np.mean((preds - val_df[target.name].to_list()) ** 2))

print(f'Validation RMSE: {rmse}')

FileNotFoundError: The system cannot find the path specified. (os error 3): ./jane_street_data/train.parquet